# 02 Analysis: Google Trends and Campaign Performance

This notebook evaluates how overall U.S. Google Trends search-interest signals align with Facebook campaign performance over the same dates.

Important scope note:
- Google Trends is used here as an external market-demand context signal, not as a user-level measure of search behavior for a specific Facebook audience segment.
- The merged dataset links theme-level search interest to campaign performance by date, so the findings should be interpreted as alignment rather than attribution or causation.

Goals:
- evaluate whether higher overall market search interest aligns with stronger campaign KPIs
- compare theme-level demand signals across CTR, conversion rate, and ROAS alignment
- identify whether age and gender segments react differently to external demand context
- document early findings carefully as exploratory rather than causal


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "processed" / "campaign_trends_daily_merged.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from the current notebook working directory.")


BASE_DIR = find_project_root()
DAILY_PATH = BASE_DIR / "data" / "processed" / "campaign_trends_daily_merged.csv"
SEGMENTED_PATH = BASE_DIR / "data" / "processed" / "campaign_trends_segmented_merged.csv"
TRENDS_PATH = BASE_DIR / "data" / "raw" / "google_trends.csv"

daily_df = pd.read_csv(DAILY_PATH, parse_dates=["date"])
segmented_df = pd.read_csv(SEGMENTED_PATH, parse_dates=["date"])
trends_df = pd.read_csv(TRENDS_PATH, parse_dates=["date"])

print("Merged campaign and trends data loaded successfully.")
print(f"Daily merged shape: {daily_df.shape}")
print(f"Segmented merged shape: {segmented_df.shape}")
print(f"Trends shape: {trends_df.shape}")

## Dataset Overview

In [ ]:
daily_df.head()

In [ ]:
segmented_df.head()

In [ ]:
theme_summary = trends_df.groupby("trend_keyword")["search_interest"].agg(["min", "max", "mean", "std", "nunique"])
theme_summary.sort_index()

## Overall Demand vs Performance

Start by checking whether higher overall market search interest aligns with stronger campaign efficiency at the daily overall level.

Note: demand buckets are created within each trend theme using within-theme terciles of daily search-interest rank. They are relative buckets, not absolute demand thresholds.


In [ ]:
overall_corr = (
    daily_df.groupby("trend_keyword")[["search_interest", "ctr", "conversion_rate", "roas"]]
    .corr()
    .reset_index()
)
overall_corr = overall_corr.loc[overall_corr["level_1"] == "search_interest", ["trend_keyword", "ctr", "conversion_rate", "roas"]]
overall_corr = overall_corr.rename(
    columns={
        "ctr": "corr_with_ctr",
        "conversion_rate": "corr_with_conversion_rate",
        "roas": "corr_with_roas",
    }
)
overall_corr

In [ ]:
daily_df["demand_bucket"] = pd.qcut(
    daily_df.groupby("trend_keyword")["search_interest"].rank(method="first"),
    q=3,
    labels=["low_demand", "mid_demand", "high_demand"],
)

bucket_summary = (
    daily_df.groupby(["trend_keyword", "demand_bucket"], as_index=False)
    .agg(
        avg_search_interest=("search_interest", "mean"),
        avg_ctr=("ctr", "mean"),
        avg_conversion_rate=("conversion_rate", "mean"),
        avg_roas=("roas", "mean"),
    )
)
bucket_summary.sort_values(["trend_keyword", "avg_search_interest"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bucket_summary.pivot(index="trend_keyword", columns="demand_bucket", values="avg_ctr").plot(
    kind="bar",
    ax=axes[0],
    title="Average CTR by Demand Bucket",
)
axes[0].set_ylabel("Average CTR")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend().remove()

bucket_summary.pivot(index="trend_keyword", columns="demand_bucket", values="avg_conversion_rate").plot(
    kind="bar",
    ax=axes[1],
    title="Average Conversion Rate by Demand Bucket",
)
axes[1].set_ylabel("Average Conversion Rate")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend().remove()

bucket_summary.pivot(index="trend_keyword", columns="demand_bucket", values="avg_roas").plot(
    kind="bar",
    ax=axes[2],
    title="Average ROAS by Demand Bucket",
)
axes[2].set_ylabel("Average ROAS")
axes[2].tick_params(axis="x", rotation=45)
axes[2].legend(title="Demand Bucket", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()


## Demand Theme Comparison

Theme-level average KPI comparisons are not shown here because the merged dataset repeats the same daily Facebook campaign metrics across all trend themes for a given date.
A more defensible comparison is to examine how each theme's overall market search-interest series aligns with campaign KPIs and how KPI averages differ across low-, mid-, and high-demand days.


In [ ]:
theme_signal_summary = (
    daily_df.groupby("trend_keyword", as_index=False)
    .agg(
        avg_search_interest=("search_interest", "mean"),
        std_search_interest=("search_interest", "std"),
        min_search_interest=("search_interest", "min"),
        max_search_interest=("search_interest", "max"),
    )
)
theme_signal_summary.sort_values("avg_search_interest", ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

theme_signal_summary.sort_values("avg_search_interest").plot(
    kind="barh",
    x="trend_keyword",
    y="avg_search_interest",
    ax=axes[0],
    color="tab:blue",
    title="Average Search Interest by Theme",
)
axes[0].set_ylabel("")

theme_signal_summary.sort_values("std_search_interest").plot(
    kind="barh",
    x="trend_keyword",
    y="std_search_interest",
    ax=axes[1],
    color="tab:purple",
    title="Search Interest Volatility by Theme",
)
axes[1].set_ylabel("")

plt.tight_layout()


## Age and Gender Sensitivity to Demand

This section evaluates whether some demographic segments show stronger positive or negative alignment with overall market search interest than others.
These results do not mean that the segment itself searched for the theme more often; they only describe whether segment performance moved with the external demand signal over time.


In [ ]:
segment_corr = (
    segmented_df.groupby(["trend_keyword", "age", "gender"])[["search_interest", "ctr", "conversion_rate", "roas"]]
    .corr()
    .reset_index()
)
segment_corr = segment_corr.loc[
    segment_corr["level_3"] == "search_interest",
    ["trend_keyword", "age", "gender", "ctr", "conversion_rate", "roas"],
]
segment_corr = segment_corr.rename(
    columns={
        "ctr": "corr_with_ctr",
        "conversion_rate": "corr_with_conversion_rate",
        "roas": "corr_with_roas",
    }
)
segment_corr.sort_values("corr_with_roas", ascending=False).head(15)

In [ ]:
segment_corr_with_context = segment_corr.merge(
    segmented_df.groupby(["trend_keyword", "age", "gender"], as_index=False).agg(
        total_spent=("spent", "sum"),
        total_clicks=("clicks", "sum"),
        total_conversions=("approved_conversion", "sum"),
        n_days=("date", "nunique"),
    ),
    on=["trend_keyword", "age", "gender"],
    how="left",
)
segment_corr_with_context.sort_values("corr_with_roas", ascending=False).head(20)


In [ ]:
segment_roas_sensitivity = segment_corr_with_context.loc[
    (segment_corr_with_context["total_spent"] >= 4000) & (segment_corr_with_context["n_days"] >= 12)
].copy()
segment_roas_sensitivity["age_gender"] = segment_roas_sensitivity["age"] + " | " + segment_roas_sensitivity["gender"]
segment_roas_sensitivity.sort_values("corr_with_roas", ascending=False).head(20)


In [ ]:
top_segment_sensitivity = segment_roas_sensitivity.sort_values("corr_with_roas", ascending=False).head(8).copy()
bottom_segment_sensitivity = segment_roas_sensitivity.sort_values("corr_with_roas", ascending=True).head(8).copy()

top_segment_sensitivity["segment_theme"] = top_segment_sensitivity["age"] + " | " + top_segment_sensitivity["gender"] + " | " + top_segment_sensitivity["trend_keyword"]
bottom_segment_sensitivity["segment_theme"] = bottom_segment_sensitivity["age"] + " | " + bottom_segment_sensitivity["gender"] + " | " + bottom_segment_sensitivity["trend_keyword"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)

top_segment_sensitivity.sort_values("corr_with_roas").plot(
    kind="barh",
    x="segment_theme",
    y="corr_with_roas",
    ax=axes[0],
    color="tab:green",
    legend=False,
    title="Most Positive Segment-Level ROAS Correlations",
)
axes[0].set_xlabel("Correlation with ROAS")
axes[0].set_ylabel("")

bottom_segment_sensitivity.sort_values("corr_with_roas").plot(
    kind="barh",
    x="segment_theme",
    y="corr_with_roas",
    ax=axes[1],
    color="tab:red",
    legend=False,
    title="Most Negative Segment-Level ROAS Correlations",
)
axes[1].set_xlabel("Correlation with ROAS")
axes[1].set_ylabel("")

plt.tight_layout()


## Time-Series Overlay by Theme

These small multiples are more readable than a pooled scatter plot because the dataset only covers 14 days.
They are intended to show whether each theme's overall market search-interest series appears to move in the same or opposite direction as campaign ROAS over time.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

for index, (keyword, group) in enumerate(daily_df.groupby("trend_keyword")):
    group = group.sort_values("date")
    ax1 = axes[index]
    ax2 = ax1.twinx()

    ax1.plot(group["date"], group["search_interest"], marker="o", color="tab:blue", label="search_interest")
    ax2.plot(group["date"], group["roas"], marker="s", color="tab:orange", label="roas")

    ax1.set_title(f"{keyword.title()}: Search Interest and ROAS Over Time")
    ax1.set_ylabel("Search Interest", color="tab:blue")
    ax2.set_ylabel("ROAS", color="tab:orange")

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.tight_layout()


## Analysis Findings

### 1. External demand signals do not align with campaign performance in the same way across all themes.

At the daily overall level, the relationship between Google Trends search interest and campaign KPIs varies meaningfully by theme. Fitness shows the clearest positive alignment with ROAS and conversion rate, while fashion and grocery show slightly negative ROAS alignment during this short observation window. Skin care appears mildly positive, but the pattern is weaker than fitness. This suggests that external market demand should not be treated as a uniform uplift signal across all topics.

### 2. Higher market search interest appears to influence post-click efficiency more than click generation.

The demand-bucket analysis suggests that stronger external demand does not consistently correspond to higher CTR. In some themes, high-demand days are associated with lower or flat CTR but stronger conversion rate and ROAS. Fitness is the clearest example: its highest-demand periods show lower average CTR but meaningfully higher conversion rate and ROAS. This indicates that demand shifts may matter more after the click than at the attention stage.

### 3. Fitness is the strongest candidate for a positive demand-performance alignment story.

Among the four manually selected themes, fitness shows the most convincing pattern of positive alignment between overall market search interest and campaign efficiency. Its correlation with ROAS is positive, and its high-demand days produce the strongest average conversion rate and ROAS for that theme. In contrast, fashion and grocery do not show the same consistent positive pattern over this time period.

### 4. Demand sensitivity differs across age and gender segments.

At the segmented level, some age and gender groups appear more positively aligned with external demand shifts than others. For example, certain female segments show stronger positive ROAS alignment under fashion and grocery demand contexts, while several male segments under fitness demand show weaker or negative ROAS alignment over the same period. These patterns suggest that external market demand may interact differently with demographic segments, even when the demand signal itself is measured at the overall market level.

### 5. These findings describe alignment with market context, not user-level search behavior.

Google Trends is used here as an overall U.S. market-demand context signal, not as a direct measure of what a specific Facebook audience segment searched for. The merged dataset links theme-level search interest to Facebook performance by date, so the results should be interpreted as evidence of temporal alignment between external demand and campaign outcomes. They should not be interpreted as proof that a given segment personally searched more for a theme or that the external demand signal caused the observed performance changes.

### 6. The analysis remains exploratory because the time window is short.

The merged dataset covers only 14 days, which is enough for directional exploration but not strong causal inference. The findings are useful for identifying potentially demand-sensitive themes and segments, but they should be framed as preliminary evidence rather than definitive business proof. A longer time window or additional external data would be needed to support stronger conclusions.

